In [ ]:
import google as genai
import time
import json
import pandas as pd

with open('config.json', 'r') as f:
    config = json.load(f)

API_KEY = config['API_KEY']

problemas_tr = pd.read_parquet("GSM8K/train-00000-of-00001.parquet")

seleccion_total = problemas_tr.sample(n=60, random_state=67)

lista_de_lotes = [seleccion_total.iloc[i:i+25] for i in range(0, 75, 25)]

print(f"Hemos creado {len(lista_de_lotes)} lotes de 25 preguntas cada uno.")

Hemos creado 3 lotes de 25 preguntas cada uno.


In [7]:
from google import genai

# --- CONFIGURACIÓN ---
client = genai.Client(api_key=API_KEY)

df_control = seleccion_total.iloc[0:20].copy()    # Grupo 1: Intacto
df_reimag  = seleccion_total.iloc[20:40].copy()   # Grupo 2: Solo cambio texto
df_hard    = seleccion_total.iloc[40:60].copy()   # Grupo 3: 5-7 pasos, aritmética dura

resultados_finales = []

# ==========================================
# 1. GRUPO CONTROL
# ==========================================
print("--- Procesando Grupo 1: Control (20 items) ---")
for idx, row in df_control.iterrows():
    resultados_finales.append({
        "id_original": idx,
        "tipo": "control",
        "problema_texto": row['question'],
        "solucion_esperada": row['answer'] 
    })
print("Grupo Control guardado.\n")


# ==========================================
# 2. GRUPO REIMAGINADO
# ==========================================
print("--- Procesando Grupo 2: Reimaginación (20 items) ---")

def prompt_reimaginado(df_chunk):
    texto_input = "\n".join([f"ID_{i}: {row['question']}" for i, row in df_chunk.iterrows()])
    
    return f"""
    You are a mathematician passionate about Science Fiction and Fantasy.
    I have a list of math problems. For EACH ONE:
    
    1. CHANGE SETTING: Rewrite the story completely. Use contexts like Cyberpunk, Dungeons & Dragons, or Space Exploration.
    2. NUMERIC CONSERVATION: The numerical result MUST remain exactly the same, but you must complicate the visual representation of numbers. 
       - Example: Instead of "25", write "5^2".
       - Example: Instead of "10", write "1000/100".
    3. OBJECTIVE: Ensure an AI model does not recognize the original text from the GSM8K dataset, but the numeric logic remains identical.
    4. LANGUAGE: The output text MUST be in ENGLISH.

    Return a JSON (list of objects):
    [
        {{ "id_original": 123, "nuevo_texto": "The full text of the reimagined problem in English..." }},
        ...
    ]
    
    PROBLEMS:
    {texto_input}
    """

try:
    response = client.models.generate_content(
        model='gemini-2.5-flash', 
        contents=prompt_reimaginado(df_reimag),
        config={'response_mime_type': 'application/json'}
    )
    data = json.loads(response.text)
    
    for item in data:
        idx_orig = item.get('id_original') 
        
        if idx_orig is not None:
            solucion_orig = df_reimag.loc[idx_orig]['answer'] 
            
            resultados_finales.append({
                "id_original": str(idx_orig),
                "tipo": "reimaginado",
                "problema_texto": item.get('nuevo_texto'),
                "solucion_esperada": solucion_orig 
            })
            
    print(f"Grupo Reimaginado completado ({len(data)} items).\n")
    time.sleep(2)

except Exception as e:
    print(f"Error en Grupo 2: {e}")

--- Procesando Grupo 1: Control (20 items) ---
Grupo Control guardado.

--- Procesando Grupo 2: Reimaginación (20 items) ---
Grupo Reimaginado completado (20 items).



In [8]:
import pandas as pd
from google import genai
import json
import time

# --- CONFIGURACIÓN PREVIA ---
# (Asegúrate de tener df_hard y client definidos como antes)
# df_hard = seleccion_total.iloc[40:60].copy() 
# resultados_finales = [] # Ojo: si ya tienes resultados de los grupos 1 y 2, no borres esta lista, solo añade.

print(f"--- Procesando Grupo 3: Hardcore (Dividido en lotes) ---")

def construir_prompt_hardcore(df_chunk):
    lines = []
    for i, row in df_chunk.iterrows():
        texto = row.get('question', row.get('pregunta', 'Error'))
        lines.append(f"ID_{i}: {texto}")
        
    texto_input = "\n".join(lines)
    
    return """
    You are an expert professor creating difficult math problems.
    For each problem listed below:
    
    1. CHANGE THE CONTEXT: Use fields like Physics, Engineering, or Economics.
    2. INCREASE DIFFICULTY: The problem MUST require 5-7 logical steps. Use "messy" numbers (decimals, fractions).
    3. SOLVE IT: Calculate the new result step-by-step.
    4. LANGUAGE: The output text MUST be in ENGLISH.

    JSON FORMAT (List of objects):
    [
        { 
            "id_original": 123, 
            "new_text": "The full text of the complex problem...", 
            "new_solution_step_by_step": "Step 1: ..., Step 2: ...", 
            "new_final_result": "The final number" 
        }
    ]
    
    PROBLEMS:
    """ + "\n" + texto_input

# --- ITERACIÓN POR LOTES (CHUNKS) ---
BATCH_SIZE = 4
total_hardcore = 0

for i in range(0, len(df_hard), BATCH_SIZE):
    df_batch = df_hard.iloc[i : i + BATCH_SIZE]
    
    print(f"   > Procesando lote {i} a {i+BATCH_SIZE}...")
    
    try:
        prompt = construir_prompt_hardcore(df_batch)
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config={'response_mime_type': 'application/json'}
        )
        
        data = json.loads(response.text)
        
        for item in data:
            raw_id = item.get('id_original')
            
            if isinstance(raw_id, str) and raw_id.startswith("ID_"):
                 raw_id = raw_id.replace("ID_", "")
            
            if raw_id is not None:
                resultados_finales.append({
                    "id_original": str(raw_id),
                    "tipo": "hardcore",
                    "problema_texto": item.get('new_text'), 
                    "solucion_esperada": f"{item.get('new_final_result')} || {item.get('new_solution_step_by_step')}"
                })
                total_hardcore += 1
        
        time.sleep(5)

    except json.JSONDecodeError:
        print(f"Error: El lote {i} devolvió un JSON roto (probablemente texto truncado).")
    except Exception as e:
        print(f"Error inesperado en lote {i}: {e}")

print(f"\nGrupo Hardcore terminado. Total procesados: {total_hardcore}")

# ==========================================
# GUARDADO FINAL
# ==========================================
df_final = pd.DataFrame(resultados_finales)

if 'id_original' in df_final.columns:
    df_final['id_original'] = df_final['id_original'].astype(str)

try:
    df_final.to_parquet("dataset_final_experimento_v2.parquet", engine="pyarrow")
    print("Guardado exitoso. Se ha convertido 'id_original' a texto para evitar conflictos.")
except Exception as e:
    print(f"Problemas guardando: {e}")

--- Procesando Grupo 3: Hardcore (Dividido en lotes) ---
   > Procesando lote 0 a 4...
   > Procesando lote 4 a 8...
   > Procesando lote 8 a 12...
   > Procesando lote 12 a 16...
   > Procesando lote 16 a 20...

Grupo Hardcore terminado. Total procesados: 20
Guardado exitoso. Se ha convertido 'id_original' a texto para evitar conflictos.


In [11]:
df_final.to_csv("Preguntas_Modelos.csv")
df_final

,id_original,tipo,problema_texto,solucion_esperada
0,3687,control,Mason has 3 cartons of 200 blueberries. He mak...,First find the total number of blueberries Mas...
1,6944,control,A local business was selling 25 raffle tickets...,They sold 25 raffle tickets at $2.00 a piece s...
2,6775,control,Iris’ family is planning a surprise birthday p...,Each of her aunts and uncles have a family uni...
3,4816,control,Mrs. Crocker made 11 pieces of fried chicken f...,"After Lyndee ate, there were 11 - 1 = <<11-1=1..."
4,4163,control,Tom buys 20 shares of a stock that costs $3 ea...,He spends 3*20=$<<3*20=60>>60 on the shares.\n...
5,4340,control,Claudia offers art classes to kids and charges...,Sunday’s class is half the size of Saturday’s ...
6,4785,control,"Every day, Lou works out by running three mile...",If Lou runs 3 miles during his workout on a tr...
7,3649,control,Uncle Ben has 440 chickens on his farm. 39 are...,Uncle Ben has 440 chickens - 39 roosters = <<4...
8,4999,control,Chastity bought 4 lollipops which cost $1.50 e...,She spent 4 x $1.50 = $<<4*1.5=6>>6 for the lo...
9,2810,control,If there are four times as many red crayons as...,Red crayons: 4(3)=12\nTotal crayons: 12+3=<<12...
